In [1]:
import pandas as pd

df_muni = pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\final_municipality_dataset.csv')
print('Municipality dataset loaded:', df_muni.shape)

Municipality dataset loaded: (1642981, 24)


In [2]:

df_district_flood = pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\final_flood_dataset_77districts.csv')
df_district_flood['level'] = 'district'
df_district_flood['municipality'] = None


flood_cols_needed = [
    'temperature', 'rainfall', 'humidity', 'wind_speed',
    'discharge', 'rainfall_roll3', 'rainfall_roll7',
    'discharge_roll3', 'discharge_roll7',
    'month', 'terrain', 'location', 'district', 'municipality',
    'date', 'flood_risk', 'flood_risk_label'
]
df_muni_flood = df_muni[flood_cols_needed].copy()
df_muni_flood['level'] = 'municipality'

print('District flood columns:', sorted(df_district_flood.columns.tolist()))
print('\nMunicipality flood columns:', sorted(df_muni_flood.columns.tolist()))

District flood columns: ['date', 'day', 'discharge', 'discharge_roll3', 'discharge_roll7', 'flood_risk', 'flood_risk_label', 'humidity', 'level', 'location', 'location_encoded', 'month', 'municipality', 'rainfall', 'rainfall_roll3', 'rainfall_roll7', 'season', 'season_encoded', 'temperature', 'terrain', 'terrain_encoded', 'wind_speed', 'year']

Municipality flood columns: ['date', 'discharge', 'discharge_roll3', 'discharge_roll7', 'district', 'flood_risk', 'flood_risk_label', 'humidity', 'level', 'location', 'month', 'municipality', 'rainfall', 'rainfall_roll3', 'rainfall_roll7', 'temperature', 'terrain', 'wind_speed']


In [3]:

df_muni_flood = df_muni[[
    'temperature', 'rainfall', 'humidity', 'wind_speed',
    'discharge', 'rainfall_roll3', 'rainfall_roll7',
    'discharge_roll3', 'discharge_roll7',
    'month', 'terrain', 'location', 'district', 'municipality',
    'date', 'flood_risk', 'flood_risk_label'
]].copy()

df_muni_flood['date'] = pd.to_datetime(df_muni_flood['date'])
df_muni_flood['day'] = df_muni_flood['date'].dt.day
df_muni_flood['year'] = df_muni_flood['date'].dt.year

def get_season(month):
    if month in [3, 4, 5]:   return 'pre_monsoon'
    elif month in [6, 7, 8, 9]: return 'monsoon'
    elif month in [10, 11]:  return 'post_monsoon'
    else:                    return 'winter'
df_muni_flood['season'] = df_muni_flood['month'].apply(get_season)
df_muni_flood['level'] = 'municipality'


df_district_flood['district'] = df_district_flood['location']
df_district_flood['level'] = 'district'
df_district_flood['municipality'] = None


df_district_flood = df_district_flood.drop(columns=['location_encoded', 'season_encoded', 'terrain_encoded'], errors='ignore')


df_flood_final = pd.concat([df_district_flood, df_muni_flood], ignore_index=True)
print(f'Combined flood dataset: {df_flood_final.shape[0]} rows, {df_flood_final["location"].nunique()} unique locations')
print(df_flood_final['level'].value_counts())

Combined flood dataset: 2374250 rows, 250 unique locations
level
municipality    1642981
district         731269
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import LabelEncoder

le_location_v2 = LabelEncoder()
df_flood_final['location_encoded'] = le_location_v2.fit_transform(df_flood_final['location'])

le_terrain_v2 = LabelEncoder()
df_flood_final['terrain_encoded'] = le_terrain_v2.fit_transform(df_flood_final['terrain'])

print('Total locations encoded:', len(le_location_v2.classes_))
print('Terrain classes:', le_terrain_v2.classes_)

print('\nFinal flood label distribution:')
print(df_flood_final['flood_risk'].value_counts())
print(df_flood_final['flood_risk'].value_counts(normalize=True).mul(100).round(2))

Total locations encoded: 250
Terrain classes: ['Hilly' 'Mountain' 'Terai']

Final flood label distribution:
flood_risk
Low       2357401
Medium      15724
High         1125
Name: count, dtype: int64
flood_risk
Low       99.29
Medium     0.66
High       0.05
Name: proportion, dtype: float64


In [5]:
df_flood_final.to_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\final_flood_dataset_with_municipalities.csv', index=False)
print('Saved!')
print(f'Shape: {df_flood_final.shape}')

Saved!
Shape: (2374250, 23)


In [6]:
df_district_landslide = pd.read_csv(r'C:\Nepal_Flood_Project\Data\final_landslide_dataset_77districts.csv')

df_muni_landslide = df_muni[[
    'temperature', 'rainfall', 'humidity', 'wind_speed',
    'rainfall_roll3', 'rainfall_roll7', 'soil_moisture', 'slope',
    'month', 'terrain', 'location', 'district', 'municipality',
    'date', 'landslide_risk', 'landslide_risk_label'
]].copy()
df_muni_landslide = df_muni_landslide.rename(columns={'rainfall_roll3': 'rain_3day', 'rainfall_roll7': 'rain_7day'})
df_muni_landslide['date'] = pd.to_datetime(df_muni_landslide['date'])
df_muni_landslide['day'] = df_muni_landslide['date'].dt.day
df_muni_landslide['year'] = df_muni_landslide['date'].dt.year
df_muni_landslide['season'] = df_muni_landslide['month'].apply(get_season)
df_muni_landslide['level'] = 'municipality'

df_district_landslide['district'] = df_district_landslide['location']
df_district_landslide['level'] = 'district'
df_district_landslide['municipality'] = None
df_district_landslide = df_district_landslide.drop(columns=['location_encoded', 'season_encoded', 'terrain_encoded'], errors='ignore')

print('District landslide columns:', sorted(df_district_landslide.columns.tolist()))
print('\nMunicipality landslide columns:', sorted(df_muni_landslide.columns.tolist()))

District landslide columns: ['date', 'day', 'district', 'humidity', 'landslide_risk', 'landslide_risk_label', 'level', 'location', 'month', 'municipality', 'rain_3day', 'rain_7day', 'rainfall', 'season', 'slope', 'soil_moisture', 'temperature', 'terrain', 'wind_speed', 'year']

Municipality landslide columns: ['date', 'day', 'district', 'humidity', 'landslide_risk', 'landslide_risk_label', 'level', 'location', 'month', 'municipality', 'rain_3day', 'rain_7day', 'rainfall', 'season', 'slope', 'soil_moisture', 'temperature', 'terrain', 'wind_speed', 'year']


In [7]:
df_landslide_final = pd.concat([df_district_landslide, df_muni_landslide], ignore_index=True)
print(f'Combined landslide dataset: {df_landslide_final.shape[0]} rows, {df_landslide_final["location"].nunique()} unique locations')
print(df_landslide_final['level'].value_counts())

Combined landslide dataset: 2374250 rows, 250 unique locations
level
municipality    1642981
district         731269
Name: count, dtype: int64


In [8]:
le_location_ls_v2 = LabelEncoder()
df_landslide_final['location_encoded'] = le_location_ls_v2.fit_transform(df_landslide_final['location'])

le_terrain_ls_v2 = LabelEncoder()
df_landslide_final['terrain_encoded'] = le_terrain_ls_v2.fit_transform(df_landslide_final['terrain'])

print('Total locations encoded:', len(le_location_ls_v2.classes_))

print('\nFinal landslide label distribution:')
print(df_landslide_final['landslide_risk'].value_counts())
print(df_landslide_final['landslide_risk'].value_counts(normalize=True).mul(100).round(2))

Total locations encoded: 250

Final landslide label distribution:
landslide_risk
Low       2328276
Medium      44032
High         1942
Name: count, dtype: int64
landslide_risk
Low       98.06
Medium     1.85
High       0.08
Name: proportion, dtype: float64


In [9]:
df_landslide_final.to_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\final_landslide_dataset_with_municipalities.csv', index=False)
print('Saved!')
print(f'Shape: {df_landslide_final.shape}')

Saved!
Shape: (2374250, 22)
